# Figure 2 | Mitochondrial phenotypes and functional representations

Cortical maps of mitochondrial density (MitoD) and mitochondrial respiratory capacity (MRC), followed by participant- and group-level associations with unique visual and unique semantic variance.

In [ ]:
from pathlib import Path
import os
import tempfile

import nibabel as nib
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable
from PIL import Image
from IPython.display import display

import cortex
import vtk
from vtk.util.numpy_support import numpy_to_vtk, numpy_to_vtkIdTypeArray

# Locate the analysis workspace. Set MITO_FMRI_ROOT explicitly when
# derivatives are stored outside this repository.
def find_analysis_root():
    configured = os.environ.get("MITO_FMRI_ROOT")
    candidates = [Path(configured)] if configured else []
    candidates += [Path.cwd(), Path.cwd() / "nsd_full_cortex", Path.cwd().parent / "nsd_full_cortex"]
    for candidate in candidates:
        if (candidate / "derivatives").exists():
            return candidate.resolve()
    raise FileNotFoundError(
        "Could not locate the analysis root. Set MITO_FMRI_ROOT to the nsd_full_cortex directory."
    )

ROOT = find_analysis_root()

PROJECT_ROOT = ROOT.parent
MITO_ROOT = PROJECT_ROOT / 'mitochondrial_analysis'
SOURCE_ROOT = MITO_ROOT / 'source_maps' / 'mosharov2025'
NEUROMAPS_ROOT = MITO_ROOT / 'neuromaps_data'
OUTPUT = ROOT / 'derivatives' / 'figure2' / 'figures'
CACHE = ROOT / 'derivatives' / 'figure2' / 'cache'
OUTPUT.mkdir(parents=True, exist_ok=True)
CACHE.mkdir(parents=True, exist_ok=True)
os.environ['NEUROMAPS_DATA'] = str(NEUROMAPS_ROOT)

PYCORTEX_SUBJECT = 'fsaverage'
N_VERTICES_HEMI = 163_842
assert PYCORTEX_SUBJECT in cortex.db.subjects

## Load cortical surfaces and mitochondrial maps

In [ ]:
from neuromaps.transforms import mni152_to_fsaverage

MAP_NAMES = ('MitoD', 'MRC')
SUPPORT_THRESHOLD = 0.5

def project_support_corrected_map(name):
    cache_path = CACHE / f'mosharov2025_{name}_fsaverage164k_support_corrected.npz'
    if cache_path.exists():
        cached = np.load(cache_path)
        return {'lh': cached['lh'], 'rh': cached['rh']}

    image = nib.load(SOURCE_ROOT / f'{name}.nii.gz')
    array = np.asarray(image.dataobj)
    support_image = nib.Nifti1Image(
        (array != 0).astype(np.float32), image.affine, image.header
    )
    projected = mni152_to_fsaverage(
        image, fsavg_density='164k', method='linear'
    )
    support = mni152_to_fsaverage(
        support_image, fsavg_density='164k', method='linear'
    )

    result = {}
    for hemi, projected_gii, support_gii in zip(('lh', 'rh'), projected, support):
        values = np.asarray(projected_gii.agg_data(), dtype=np.float64).squeeze()
        weights = np.asarray(support_gii.agg_data(), dtype=np.float64).squeeze()
        keep = np.isfinite(values) & np.isfinite(weights) & (weights > SUPPORT_THRESHOLD)
        corrected = np.full(values.shape, np.nan, dtype=np.float32)
        corrected[keep] = values[keep] / weights[keep]
        result[hemi] = corrected
    np.savez_compressed(cache_path, lh=result['lh'], rh=result['rh'])
    return result

mito_maps = {name: project_support_corrected_map(name) for name in MAP_NAMES}
for name in MAP_NAMES:
    assert mito_maps[name]['lh'].shape == (N_VERTICES_HEMI,)

# Define the color and range of each phenotype once for use across all surface views.
MAP_SPECS = {
    'MitoD': {'label': 'MitoD', 'cmap': 'magma'},
    'MRC': {'label': 'MRC', 'cmap': 'viridis'},
}
MITOMAP_VMIN = 1.0
MITOMAP_VMAX = 1.2

for name in MAP_NAMES:
    valid = mito_maps[name]['lh'][np.isfinite(mito_maps[name]['lh'])]
    MAP_SPECS[name]['vmin'] = MITOMAP_VMIN
    MAP_SPECS[name]['vmax'] = MITOMAP_VMAX
    print(
        f"{name}: n={len(valid):,}, range={valid.min():.3f}–{valid.max():.3f}, "
        f"display={MAP_SPECS[name]['vmin']:.3f}–{MAP_SPECS[name]['vmax']:.3f}"
    )

## Flat-surface maps of MitoD and MRC

In [ ]:
def crop_left_flatmap(source, destination, padding=24):
    image = Image.open(source).convert('RGBA')
    left = image.crop((0, 0, int(0.98 * image.width // 2), image.height))
    rgb = np.asarray(left.convert('RGB'))
    content = np.any(rgb < 248, axis=2)
    yy, xx = np.where(content)
    box = (
        max(0, int(xx.min()) - padding), max(0, int(yy.min()) - padding),
        min(left.width, int(xx.max()) + padding + 1),
        min(left.height, int(yy.max()) + padding + 1),
    )
    left.crop(box).save(destination)

def render_flatmap(name):
    spec = MAP_SPECS[name]
    left = mito_maps[name]['lh']
    bilateral = np.concatenate([left, np.full(N_VERTICES_HEMI, np.nan)])
    vertex = cortex.Vertex(
        bilateral, PYCORTEX_SUBJECT, cmap=spec['cmap'],
        vmin=spec['vmin'], vmax=spec['vmax'],
    )
    figure = cortex.quickflat.make_figure(
        vertex, with_curvature=True, with_rois=False, with_labels=False,
        with_colorbar=False, with_borders=False, recache=False, nanmean=True,
        height=1600, curvature_brightness=0.70, curvature_contrast=0.15,
    )
    with tempfile.TemporaryDirectory() as temporary:
        full_path = Path(temporary) / 'bilateral.png'
        figure.savefig(
            full_path, dpi=200, bbox_inches='tight', pad_inches=0,
            facecolor='white', transparent=False,
        )
        plt.close(figure)
        output_path = OUTPUT / f'{name}_lh_surface.png'
        crop_left_flatmap(full_path, output_path)
    return output_path

surface_exports = {name: render_flatmap(name) for name in MAP_NAMES}
for name in MAP_NAMES:
    print(name)
    display(Image.open(surface_exports[name]))

## Lateral inflated-surface views

In [ ]:
VIEW_SIZE = 1500
inflated_lh, inflated_faces = cortex.db.get_surf(
    PYCORTEX_SUBJECT, 'inflated', hemisphere='left'
)
curvature_lh = np.asarray(
    cortex.db.get_surfinfo(PYCORTEX_SUBJECT, type='curvature').data[:N_VERTICES_HEMI]
)

def vtk_polydata(points, faces):
    poly = vtk.vtkPolyData()
    vtk_points = vtk.vtkPoints()
    vtk_points.SetData(numpy_to_vtk(np.asarray(points, np.float32), deep=True))
    poly.SetPoints(vtk_points)
    packed = np.column_stack([
        np.full(len(faces), 3, dtype=np.int64), np.asarray(faces, dtype=np.int64)
    ]).ravel()
    cells = vtk.vtkCellArray()
    cells.SetCells(len(faces), numpy_to_vtkIdTypeArray(packed, deep=True))
    poly.SetPolys(cells)
    return poly

def surface_rgb(name):
    spec = MAP_SPECS[name]
    values = mito_maps[name]['lh']
    valid = np.isfinite(values)
    norm = Normalize(spec['vmin'], spec['vmax'], clip=True)
    mapped = plt.colormaps[spec['cmap']](norm(np.nan_to_num(values, nan=spec['vmin'])))[:, :3]
    curvature_binary = (curvature_lh > 0).astype(float)
    gray = np.clip(0.70 + (curvature_binary - 0.5) * 0.15, 0, 1)
    background = np.repeat(gray[:, None], 3, axis=1)
    rgb = background.copy()
    rgb[valid] = 0.90 * mapped[valid] + 0.10 * background[valid]
    return np.clip(255 * rgb, 0, 255).astype(np.uint8)

def crop_transparent_background(path, padding=24):
    image = Image.open(path).convert('RGBA')
    rgba = np.asarray(image)
    yy, xx = np.where(rgba[:, :, 3] > 0)
    box = (
        max(0, int(xx.min()) - padding), max(0, int(yy.min()) - padding),
        min(image.width, int(xx.max()) + padding + 1),
        min(image.height, int(yy.max()) + padding + 1),
    )
    image.crop(box).save(path)

def render_lateral(name):
    poly = vtk_polydata(inflated_lh, inflated_faces)
    colors = numpy_to_vtk(surface_rgb(name), deep=True, array_type=vtk.VTK_UNSIGNED_CHAR)
    colors.SetName('surface_rgb')
    colors.SetNumberOfComponents(3)
    poly.GetPointData().SetScalars(colors)

    normals = vtk.vtkPolyDataNormals()
    normals.SetInputData(poly)
    normals.SplittingOff(); normals.ConsistencyOn(); normals.AutoOrientNormalsOn()
    mapper = vtk.vtkPolyDataMapper()
    mapper.SetInputConnection(normals.GetOutputPort())
    mapper.SetColorModeToDirectScalars()
    mapper.SetScalarModeToUsePointData()
    mapper.InterpolateScalarsBeforeMappingOn()
    actor = vtk.vtkActor(); actor.SetMapper(mapper)
    actor.GetProperty().SetAmbient(0.78)
    actor.GetProperty().SetDiffuse(0.22)
    actor.GetProperty().SetSpecular(0.0)

    renderer = vtk.vtkRenderer()
    renderer.SetBackground(1.0, 1.0, 1.0)
    renderer.SetBackgroundAlpha(0.0)
    renderer.AddActor(actor)
    window = vtk.vtkRenderWindow()
    window.SetOffScreenRendering(1); window.SetAlphaBitPlanes(1)
    window.SetSize(VIEW_SIZE, VIEW_SIZE); window.SetMultiSamples(8)
    window.AddRenderer(renderer)

    center = np.asarray(poly.GetCenter())
    bounds = np.asarray(poly.GetBounds()).reshape(3, 2)
    radius = float(np.max(bounds[:, 1] - bounds[:, 0]) / 2)
    camera = renderer.GetActiveCamera()
    camera.SetFocalPoint(*center)
    camera.SetPosition(*(center + np.array([-1.0, 0.0, 0.0]) * radius * 4.0))
    camera.SetViewUp(0.0, 0.0, 1.0)
    camera.ParallelProjectionOn(); camera.SetParallelScale(radius * 1.08)
    renderer.ResetCameraClippingRange(); window.Render()

    capture = vtk.vtkWindowToImageFilter()
    capture.SetInput(window); capture.SetInputBufferTypeToRGBA()
    capture.ReadFrontBufferOff(); capture.Update()
    path = OUTPUT / f'{name}_lh_lateral.png'
    writer = vtk.vtkPNGWriter()
    writer.SetFileName(str(path)); writer.SetInputConnection(capture.GetOutputPort())
    writer.Write(); window.Finalize()
    crop_transparent_background(path)
    return path

lateral_exports = {name: render_lateral(name) for name in MAP_NAMES}
for name in MAP_NAMES:
    print(name)
    display(Image.open(lateral_exports[name]))

## Color keys

In [ ]:
colorbar_exports = {}
for name in MAP_NAMES:
    spec = MAP_SPECS[name]
    fig, ax = plt.subplots(figsize=(4, 1), constrained_layout=True)
    sm = ScalarMappable(Normalize(spec['vmin'], spec['vmax']), cmap=spec['cmap'])
    cbar = fig.colorbar(sm, cax=ax, orientation='horizontal')
    #cbar.set_label(spec['label'], fontsize=12)
    cbar.set_ticks(np.linspace(spec['vmin'], spec['vmax'], 2))
    cbar.ax.tick_params(labelsize=30)
    path = OUTPUT / f'{name}_colorbar.png'
    fig.savefig(path, dpi=300, bbox_inches='tight', transparent=True)
    plt.close(fig)
    colorbar_exports[name] = path

for name in MAP_NAMES:
    print(f'\n{name}')
    print(surface_exports[name])
    print(lateral_exports[name])
    print(colorbar_exports[name])
    display(Image.open(surface_exports[name]))
    display(Image.open(lateral_exports[name]))
    display(Image.open(colorbar_exports[name]))

## Analysis-mask visualizations

The outline and masked-map variants use the participant-specific encoding significance mask selected for the displayed analysis.

In [ ]:
MASK_ROOT = ROOT / 'derivatives' / 'encoding_significance' / 'masks'
MASK_SUBJECT = 'subj07'
MASK_MODELS = ('dinov2_minilm', 'cornet_s_mpnet')

subj07_model_masks = {
    model: np.load(MASK_ROOT / f'{MASK_SUBJECT}_{model}_encoding_mask.npz')['mask_lh'].astype(bool)
    for model in MASK_MODELS
}
SUBJ07_ANALYSIS_MASK = np.logical_or.reduce(list(subj07_model_masks.values()))
SUBJ07_COMMON_MASK = np.logical_and.reduce(list(subj07_model_masks.values()))
assert SUBJ07_ANALYSIS_MASK.shape == (N_VERTICES_HEMI,)

print(f"DINOv2–MiniLM mask: {subj07_model_masks['dinov2_minilm'].sum():,} vertices")
print(f"CORnet-S–MPNet mask: {subj07_model_masks['cornet_s_mpnet'].sum():,} vertices")
print(f"Intersection: {SUBJ07_COMMON_MASK.sum():,} vertices")
print(f"Analysis coverage (union; used below): {SUBJ07_ANALYSIS_MASK.sum():,} vertices")

In [ ]:
from cortex.quickflat.utils import make_flatmap_image

MASK_OUTLINE_COLOR = '#707070'
MASK_OUTLINE_WIDTH_FLAT = 2.6

def render_masked_flatmap(name, variant):
    if variant not in ('mask_outline', 'mask_only'):
        raise ValueError(variant)
    spec = MAP_SPECS[name]
    left = mito_maps[name]['lh'].copy()
    if variant == 'mask_only':
        left[~SUBJ07_ANALYSIS_MASK] = np.nan
    bilateral = np.concatenate([left, np.full(N_VERTICES_HEMI, np.nan)])
    vertex = cortex.Vertex(
        bilateral, PYCORTEX_SUBJECT, cmap=spec['cmap'],
        vmin=spec['vmin'], vmax=spec['vmax'],
    )
    figure = cortex.quickflat.make_figure(
        vertex, with_curvature=True, with_rois=False, with_labels=False,
        with_colorbar=False, with_borders=False, recache=False, nanmean=True,
        height=1600, curvature_brightness=0.70, curvature_contrast=0.15,
    )
    if variant == 'mask_outline':
        bilateral_mask = np.concatenate([
            SUBJ07_ANALYSIS_MASK.astype(float),
            np.full(N_VERTICES_HEMI, np.nan),
        ])
        mask_vertex = cortex.Vertex(bilateral_mask, PYCORTEX_SUBJECT, vmin=0, vmax=1)
        mask_image, extents = make_flatmap_image(
            mask_vertex, height=1600, recache=False, nanmean=True
        )
        figure.axes[0].contour(
            np.nan_to_num(mask_image, nan=0.0), levels=[0.5],
            colors=[MASK_OUTLINE_COLOR], linewidths=MASK_OUTLINE_WIDTH_FLAT,
            extent=extents, origin='upper', zorder=30,
        )
    with tempfile.TemporaryDirectory() as temporary:
        full_path = Path(temporary) / 'bilateral.png'
        figure.savefig(
            full_path, dpi=200, bbox_inches='tight', pad_inches=0,
            facecolor='white', transparent=False,
        )
        plt.close(figure)
        output_path = OUTPUT / f'{name}_lh_surface_subj07_{variant}.png'
        crop_left_flatmap(full_path, output_path)
    return output_path

masked_surface_exports = {
    (name, variant): render_masked_flatmap(name, variant)
    for name in MAP_NAMES for variant in ('mask_outline', 'mask_only')
}
for name in MAP_NAMES:
    for variant in ('mask_outline', 'mask_only'):
        print(name, variant)
        display(Image.open(masked_surface_exports[(name, variant)]))

In [ ]:
MASK_OUTLINE_WIDTH_3D = 4.0

def surface_rgb_mask_variant(name, variant):
    spec = MAP_SPECS[name]
    values = mito_maps[name]['lh']
    valid = np.isfinite(values)
    if variant == 'mask_only':
        valid &= SUBJ07_ANALYSIS_MASK
    norm = Normalize(spec['vmin'], spec['vmax'], clip=True)
    mapped = plt.colormaps[spec['cmap']](
        norm(np.nan_to_num(values, nan=spec['vmin']))
    )[:, :3]
    curvature_binary = (curvature_lh > 0).astype(float)
    gray = np.clip(0.70 + (curvature_binary - 0.5) * 0.15, 0, 1)
    background = np.repeat(gray[:, None], 3, axis=1)
    rgb = background.copy()
    rgb[valid] = 0.90 * mapped[valid] + 0.10 * background[valid]
    return np.clip(255 * rgb, 0, 255).astype(np.uint8)

def mask_boundary_polydata(points, faces, mask):
    faces = np.asarray(faces, dtype=np.int64)
    edges = np.vstack([faces[:, [0, 1]], faces[:, [1, 2]], faces[:, [2, 0]]])
    edges = np.sort(edges, axis=1)
    edges = np.unique(edges, axis=0)
    boundary = edges[mask[edges[:, 0]] != mask[edges[:, 1]]]

    points = np.asarray(points, dtype=np.float32)
    center = points.mean(axis=0)
    lifted = center + 1.0025 * (points - center)
    poly = vtk.vtkPolyData()
    vtk_points = vtk.vtkPoints()
    vtk_points.SetData(numpy_to_vtk(lifted, deep=True))
    poly.SetPoints(vtk_points)
    packed = np.column_stack([
        np.full(len(boundary), 2, dtype=np.int64), boundary
    ]).ravel()
    lines = vtk.vtkCellArray()
    lines.SetCells(len(boundary), numpy_to_vtkIdTypeArray(packed, deep=True))
    poly.SetLines(lines)
    return poly

def render_masked_lateral(name, variant):
    if variant not in ('mask_outline', 'mask_only'):
        raise ValueError(variant)
    poly = vtk_polydata(inflated_lh, inflated_faces)
    colors = numpy_to_vtk(
        surface_rgb_mask_variant(name, variant), deep=True,
        array_type=vtk.VTK_UNSIGNED_CHAR,
    )
    colors.SetName('surface_rgb'); colors.SetNumberOfComponents(3)
    poly.GetPointData().SetScalars(colors)
    normals = vtk.vtkPolyDataNormals()
    normals.SetInputData(poly)
    normals.SplittingOff(); normals.ConsistencyOn(); normals.AutoOrientNormalsOn()
    mapper = vtk.vtkPolyDataMapper()
    mapper.SetInputConnection(normals.GetOutputPort())
    mapper.SetColorModeToDirectScalars(); mapper.SetScalarModeToUsePointData()
    mapper.InterpolateScalarsBeforeMappingOn()
    actor = vtk.vtkActor(); actor.SetMapper(mapper)
    actor.GetProperty().SetAmbient(0.78); actor.GetProperty().SetDiffuse(0.22)
    actor.GetProperty().SetSpecular(0.0)

    renderer = vtk.vtkRenderer()
    renderer.SetBackground(1.0, 1.0, 1.0); renderer.SetBackgroundAlpha(0.0)
    renderer.AddActor(actor)
    if variant == 'mask_outline':
        boundary_poly = mask_boundary_polydata(
            inflated_lh, inflated_faces, SUBJ07_ANALYSIS_MASK
        )
        boundary_mapper = vtk.vtkPolyDataMapper()
        boundary_mapper.SetInputData(boundary_poly)
        boundary_actor = vtk.vtkActor(); boundary_actor.SetMapper(boundary_mapper)
        boundary_actor.GetProperty().SetColor(0.44, 0.44, 0.44)
        boundary_actor.GetProperty().SetLineWidth(MASK_OUTLINE_WIDTH_3D)
        boundary_actor.GetProperty().SetAmbient(1.0)
        boundary_actor.GetProperty().SetDiffuse(0.0)
        renderer.AddActor(boundary_actor)

    window = vtk.vtkRenderWindow()
    window.SetOffScreenRendering(1); window.SetAlphaBitPlanes(1)
    window.SetSize(VIEW_SIZE, VIEW_SIZE); window.SetMultiSamples(8)
    window.AddRenderer(renderer)
    center = np.asarray(poly.GetCenter())
    bounds = np.asarray(poly.GetBounds()).reshape(3, 2)
    radius = float(np.max(bounds[:, 1] - bounds[:, 0]) / 2)
    camera = renderer.GetActiveCamera()
    camera.SetFocalPoint(*center)
    camera.SetPosition(*(center + np.array([-1.0, 0.0, 0.0]) * radius * 4.0))
    camera.SetViewUp(0.0, 0.0, 1.0)
    camera.ParallelProjectionOn(); camera.SetParallelScale(radius * 1.08)
    renderer.ResetCameraClippingRange(); window.Render()
    capture = vtk.vtkWindowToImageFilter()
    capture.SetInput(window); capture.SetInputBufferTypeToRGBA()
    capture.ReadFrontBufferOff(); capture.Update()
    path = OUTPUT / f'{name}_lh_lateral_subj07_{variant}.png'
    writer = vtk.vtkPNGWriter()
    writer.SetFileName(str(path)); writer.SetInputConnection(capture.GetOutputPort())
    writer.Write(); window.Finalize()
    crop_transparent_background(path)
    return path

masked_lateral_exports = {
    (name, variant): render_masked_lateral(name, variant)
    for name in MAP_NAMES for variant in ('mask_outline', 'mask_only')
}
for name in MAP_NAMES:
    for variant in ('mask_outline', 'mask_only'):
        print(name, variant)
        display(Image.open(masked_lateral_exports[(name, variant)]))

## Exported figure assets

In [ ]:
for name in MAP_NAMES:
    for variant in ('mask_outline', 'mask_only'):
        print(f'\n{name} | {variant}')
        print(masked_surface_exports[(name, variant)])
        print(masked_lateral_exports[(name, variant)])
        display(Image.open(masked_surface_exports[(name, variant)]))
        display(Image.open(masked_lateral_exports[(name, variant)]))

## Group-level mitochondrial associations

In [ ]:
import pandas as pd
from matplotlib.lines import Line2D
from matplotlib.ticker import FormatStrFormatter

MITO_RESULTS = ROOT / 'derivatives' / 'mitochondrial_analysis'
subject_stats = pd.read_csv(MITO_RESULTS / 'subject_results_100k.csv')
group_stats = pd.read_csv(MITO_RESULTS / 'group_results_100k.csv')

ANALYSES = ('dinov2_minilm', 'cornet_s_mpnet')
ANALYSIS_LABELS = {
    'dinov2_minilm': 'DINOv2 + MiniLM',
    'cornet_s_mpnet': 'CORnet-S + MPNet',
}
VARIANCE_MAPS = ('unique_visual', 'unique_semantic')
VARIANCE_STYLE = {
    'unique_visual': {'label': 'Unique visual variance', 'color': '#2369C8'},
    'unique_semantic': {'label': 'Unique semantic variance', 'color': '#DC3246'},
}
SUBJECTS = ('subj01', 'subj02', 'subj05', 'subj07')
SUBJECT_MARKERS = dict(zip(SUBJECTS, ('o', 's', '^', 'D')))
MITO_MAPS = ('MitoD', 'MRC')

ypos_base = np.arange(len(MITO_MAPS), dtype=float)
offsets = {'unique_visual': -0.13, 'unique_semantic': 0.13}

for analysis in ANALYSES:
    fig, ax = plt.subplots(figsize=(6, 3.2))
    for variance_map in VARIANCE_MAPS:
        style = VARIANCE_STYLE[variance_map]
        for xi, mito_map in enumerate(MITO_MAPS):
            rows = subject_stats.query(
                'analysis == @analysis and variance_map == @variance_map and mitochondrial_map == @mito_map'
            ).set_index('subject')
            gx = group_stats.query(
                'analysis == @analysis and variance_map == @variance_map and mitochondrial_map == @mito_map'
            ).iloc[0]
            ypos = ypos_base[xi] + offsets[variance_map]
            for subject in SUBJECTS:
                ax.scatter(
                    rows.loc[subject, 'partial_r_pg1'], ypos, s=34,
                    marker=SUBJECT_MARKERS[subject], facecolor=style['color'],
                    edgecolor='white', linewidth=0.55, alpha=0.72, zorder=3,
                )
            mean = gx['fisher_mean_partial_r']
            lo = gx['partial_bootstrap_ci_low']
            hi = gx['partial_bootstrap_ci_high']
            ax.errorbar(
                mean, ypos, xerr=[[mean - lo], [hi - mean]], fmt='D',
                ms=7.5, mfc=style['color'], mec='black', mew=0.7,
                ecolor='black', elinewidth=1.25, capsize=3.5, zorder=5,
            )
            q = gx['partial_group_spatial_q_6tests']
            if mean < 0:
                ax.text(lo - 0.018, ypos, f'q={q:.3f}', ha='right', va='center', fontsize=10)
            else:
                ax.text(hi + 0.018, ypos, f'q={q:.3f}', ha='left', va='center', fontsize=10)

    ax.axvline(0, color="#4F4F4F", linewidth=0.9, linestyle='--', zorder=1)
    ax.set_yticks(ypos_base, MITO_MAPS)
    ax.set_ylim(-0.48, 1.48)
    ax.invert_yaxis()
    ax.set_xlim(-0.36, 0.32)
    ax.xaxis.set_major_formatter(FormatStrFormatter('%.2f'))
    ax.axhspan(0.5, 1.48, color="#39B976", alpha=0.1, edgecolor='none')
    ax.axhspan(0.5, -0.48, color="#FEAE77", alpha=0.1, edgecolor='none')
    ax.set_title(ANALYSIS_LABELS[analysis], fontsize=14, pad=10, fontweight='bold')
    ax.spines[['top', 'right']].set_visible(False)
    ax.grid(axis='x', color='#E5E5E5', linewidth=0.7, zorder=0)

    ax.set_xlabel('PG1-adjusted Partial Correlation', fontsize=14)
    ax.set_ylabel('Mitochondrial Map', fontsize=14)
    variance_handles = [
        Line2D([0], [0], marker='D', linestyle='none', markersize=7,
               markerfacecolor=VARIANCE_STYLE[m]['color'], markeredgecolor='black',
               label=VARIANCE_STYLE[m]['label']) for m in VARIANCE_MAPS
    ]
    subject_handles = [
        Line2D([0], [0], marker=SUBJECT_MARKERS[s], linestyle='none', markersize=6,
               markerfacecolor='#555555', markeredgecolor='white', label=s.replace('subj', 'NSD Sub-'))
        for s in SUBJECTS
    ]
    fig.legend(handles=variance_handles + subject_handles, loc='lower center', ncol=3,
               frameon=False, bbox_to_anchor=(0.5, -0.08), fontsize=8.2)
    fig.tight_layout(rect=(0, 0.16, 1, 1))
    stem = f'group_level_{analysis}_uv_us_pg1_partial'
    group_png = OUTPUT / f'{stem}.png'
    group_pdf = OUTPUT / f'{stem}.pdf'
    fig.savefig(group_png, dpi=450, bbox_inches='tight', facecolor='white')
    fig.savefig(group_pdf, bbox_inches='tight', facecolor='white')
    plt.show()
    print(group_png)

## Participant-level vertex-wise associations

In [ ]:
masked_vectors = np.load(MITO_RESULTS / 'cache' / 'subject_masked_vectors_fsaverage10k.npz')

def residual_z(values, covariate):
    values = np.asarray(values, dtype=float)
    covariate = np.asarray(covariate, dtype=float)
    design = np.column_stack([np.ones(values.size), covariate])
    residual = values - design @ np.linalg.lstsq(design, values, rcond=None)[0]
    return (residual - residual.mean()) / residual.std(ddof=1)

def plot_subj07_scatter_grid(analysis):
    subject = 'subj07'
    pg1 = masked_vectors[f'hierarchy_{subject}_{analysis}']
    fig, axes = plt.subplots(2, 2, figsize=(8.0, 7.0), sharex=False, sharey=False)
    for row, variance_map in enumerate(VARIANCE_MAPS):
        y = residual_z(masked_vectors[f'func_{variance_map}_{subject}_{analysis}'], pg1)
        for col, mito_map in enumerate(MITO_MAPS):
            ax = axes[row, col]
            xval = residual_z(masked_vectors[f'mito_{mito_map}_{subject}_{analysis}'], pg1)
            fit = np.polyfit(xval, y, 1)
            line_x = np.linspace(np.nanpercentile(xval, 0.5), np.nanpercentile(xval, 99.5), 200)
            ax.scatter(xval, y, s=6, color=VARIANCE_STYLE[variance_map]['color'],
                       alpha=0.18, linewidths=0, rasterized=True)
            ax.plot(line_x, np.polyval(fit, line_x), color='#35125A', linewidth=2.0)
            stat = subject_stats.query(
                'subject == @subject and analysis == @analysis and variance_map == @variance_map and mitochondrial_map == @mito_map'
            ).iloc[0]
            ax.text(0.04, 0.96,
                    f"partial $r$ = {stat['partial_r_pg1']:.3f}\n"
                    f"$p$ = {stat['partial_spatial_p_100k']:.4f}\n",
                    transform=ax.transAxes, ha='left', va='top', fontsize=9,
                    bbox=dict(facecolor='white', edgecolor='none', alpha=0.82, pad=2.5))
            ax.axhline(0, color='#BDBDBD', linewidth=0.65, zorder=0)
            ax.axvline(0, color='#BDBDBD', linewidth=0.65, zorder=0)
            ax.set_yticks([-2, 0, 2, 4, 6])
            ax.set_xlabel(f'{mito_map} residual (z)', fontsize=12)
            ax.set_ylabel(f"{VARIANCE_STYLE[variance_map]['label']} residual (z)", fontsize=12)
            ax.set_title(f"{VARIANCE_STYLE[variance_map]['label']} vs {mito_map}", fontsize=10.5)
            ax.spines[['top', 'right']].set_visible(False)
    fig.suptitle(f"{ANALYSIS_LABELS[analysis]}", fontsize=20, y=0.98, fontweight='bold')
    fig.tight_layout()
    stem = f'subj07_{analysis}_uv_us_pg1_partial_scatter_grid'
    png = OUTPUT / f'{stem}.png'
    pdf = OUTPUT / f'{stem}.pdf'
    fig.savefig(png, dpi=450, bbox_inches='tight', facecolor='white')
    fig.savefig(pdf, bbox_inches='tight', facecolor='white')
    plt.show()
    return png

subj07_scatter_exports = {analysis: plot_subj07_scatter_grid(analysis) for analysis in ANALYSES}
subj07_scatter_exports